### Tools 
Tools let agents take actions: things like fetching data, running code, calling external APIs, and even using a computer. The SDK supports five categories:

[Documentation](https://openai.github.io/openai-agents-python/tools/)

- **Hosted OpenAI tools**: execute for the model on OpenAI servers.
- **Local/runtime execution tools**: ComputerTool and ApplyPatchTool always run in your environment, while ShellTool can run locally or in a hosted container.
- **FunctionTool instances**: wrap any Python function as a tool.
- **Agents as tools**: expose an agent as a callable tool without a full handoff.
- **Experimental**: **Codex tool**: run workspace-scoped Codex tasks from a tool call.


In [7]:
from agents import (Agent, 
                    Runner, AsyncOpenAI, 
                    OpenAIChatCompletionsModel, set_tracing_disabled)

from agents.decorators import tool

In [2]:
# Tracing disabled
set_tracing_disabled(disabled=True)

In [3]:
import os
gemini_api = os.environ['GEMINI_API_KEY']

In [4]:
# 1. Which LLM Service?
external_client = AsyncOpenAI(
    api_key=gemini_api,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [15]:
# 2. Which LLM Model?
llm_model = OpenAIChatCompletionsModel(
    model="gemini-3.1-flash-lite",
    openai_client=external_client
)

### Tool creation

In [16]:
@tool
def get_weather(city: str) -> str:
    """Return the current weather of a given city."""
    return f"It's sunny in {city} today."

In [17]:
# Agent Creation
weather_agent: Agent = Agent(name="Weather Agent",
                     instructions="You are a helpful assistant.",
                     model=llm_model,
                     tools = [get_weather])

In [18]:
result = await Runner.run(weather_agent, "What is the Current weather in Lahore?")

print("\nCALLING AGENT\n")
print(result.final_output)


CALLING AGENT

The current weather in Lahore is sunny.


In [19]:
from pprint import pprint
pprint(result)

RunResult(input='What is the Current weather in Lahore?',
          new_items=[ToolCallItem(agent=Agent(name='Weather Agent',
                                              handoff_description=None,
                                              tools=[FunctionTool(name='get_weather',
                                                                  description='Return '
                                                                              'the '
                                                                              'current '
                                                                              'weather '
                                                                              'of '
                                                                              'a '
                                                                              'given '
                                                                              'city.',
                        